# Elastic path morphing
This example uses NeuroPath2Path matching and the numerical interpolation from the authors' earlier [ElasticPath2Path tool](https://github.com/50-Cent/ElasticPath2Path/blob/master/visualMorphingNeuron.m). It reads two granule neurons from the supplied archive.

Intermediate results are collections of independently evolving paths. They do not define a connected branching SWC skeleton, and do not interpolate radii. The original method's reconstruction correction can allow intermediate roots to drift.

In [1]:
from pathlib import Path
from io import StringIO
from zipfile import ZipFile
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'Intercellular_data.zip').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from neuropath2path import read_swc, extract_features, morph_neurons, write_morph_html

In [2]:

with ZipFile(ROOT / 'Intercellular_data.zip') as archive:
    names = sorted(name for name in archive.namelist()
                   if '/Granule/' in name and name.endswith('.swc'))[:2]
    first, second = [read_swc(StringIO(archive.read(name).decode('utf-8-sig')))
                     for name in names]
for name, neuron in zip(names, (first, second)):
    print(name, '—', extract_features(neuron).path_count, 'dendritic paths')

# To use your own morphologies instead:
# first = read_swc('/path/to/first.swc')
# second = read_swc('/path/to/second.swc')

Cell_type/Granule/cho/CNG version/L0578-slide-2-section-6-left-1.CNG.swc — 12 dendritic paths
Cell_type/Granule/cho/CNG version/L0579-slide-2-section-5-left-side-1.CNG.swc — 13 dendritic paths


In [7]:
import os, sys
from pathlib import Path

morpho_path = '/home/timonidis/PycharmProjects/TopographyMapper/src/topography_mapper/saves/morphologies/seu_allen'
first = read_swc(os.path.join(morpho_path, os.listdir(morpho_path)[0]))
second = read_swc(os.path.join(morpho_path, os.listdir(morpho_path)[1]))




In [9]:
morph = morph_neurons(first, second, times=np.linspace(0, 1, 21),
                     samples=128, method='paper', axis_order='original')
print(len(morph.paths), 'matched pairs; first pair:', morph.pairs[0])
print('Frame coordinates per pair:', morph.paths[0].frames.shape)
print('Additional pairs to retain endpoint coverage:', morph.added_pairs.tolist())
html_path = write_morph_html(morph, ROOT / 'results' / 'path_morph.html')
print('Standalone animation:', html_path)

50 matched pairs; first pair: [ 0 36]
Frame coordinates per pair: (21, 128, 3)
Additional pairs to retain endpoint coverage: []
Standalone animation: /home/timonidis/PycharmProjects/NeuroPath2Path/results/path_morph.html


In [10]:
from html import escape
from IPython.display import HTML, display
# Embed the standalone viewer without needing a local web server.
document = escape(html_path.read_text(encoding='utf-8'), quote=True)
display(HTML(f'<iframe srcdoc="{document}" width="100%" height="750" style="border:0"></iframe>'))

For the interval-based discretization of NeuroPath2Path §3.4, use `method='paper'`. This variant anchors all roots at the origin. Use `axis_order='range'` for the descending-range coordinate permutation described in Fig. 6.

A finite `samples` may remove original bends during resampling. With `samples=None`, the earlier-tool backend upsamples to the larger input size; the paper backend uses the union of the two normalized arc-length knot sets, preserving both endpoint polylines. The intermediate frames generally differ between the two discretizations.